In [20]:
import pandas as pd
import numpy as np
dataset=pd.read_csv("/content/spam.csv", encoding='latin-1')
dataset

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [22]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
ps=PorterStemmer()
nltk.download("stopwords")
all_stopword=set(stopwords.words("english"))
def clean_dataset (text):
 text=re.sub(r'<br/s*?>',' ',text)
 text=re.sub('[^a-zA-Z]',' ',text)
 words=text.split()
 text=text.lower()
 clean_V2=[ps.stem(word) for word in words if not word in all_stopword]
 return ' '.join(clean_V2)
dataset['cleanedV2']=dataset["v2"].apply(clean_dataset)
display(dataset[['v2','cleanedV2']].head())


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,v2,cleanedV2
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah i think goe usf live around though


In [51]:
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection  import train_test_split
from sklearn.naive_bayes import MultinomialNB
tfidf=TfidfVectorizer(max_features=1000)
classifier=MultinomialNB()
X=tfidf.fit_transform(dataset['cleanedV2'])
y=dataset['v1'].map({'ham':1,'spam':0})
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=50)
classifier.fit(X_train,y_train)
y_pred=classifier.predict(X_test)
y1_pred=classifier.predict(X_train)
test=accuracy_score(y_test,y_pred)
train=accuracy_score(y_train,y1_pred)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['ham', 'spam']))







Classification Report:
               precision    recall  f1-score   support

         ham       0.97      0.81      0.89       161
        spam       0.97      1.00      0.98       954

    accuracy                           0.97      1115
   macro avg       0.97      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [49]:
def prediction_dataset (custom_text):
  clean_set=clean_dataset(custom_text)
  vectorize=tfidf.transform([clean_set])
  prediction=classifier.predict(vectorize)[0]
  if(prediction==1):
    v2="ham"
  else:
    v2="spam"
  print(f"v2={custom_text}\n")
  print(f"type:{v2}")
prediction_dataset("free entry in 2 a wkly comp to win FA Cup final")
prediction_dataset("Hey, are we still meeting up for dinner at 7 PM tonight? Let me know if you are free.")
prediction_dataset("URGENT SECURITY NOTICE: Your banking application has been temporarily locked due to suspicious activity. Please login here to verify your identity.")








v2=free entry in 2 a wkly comp to win FA Cup final

type:spam
v2=Hey, are we still meeting up for dinner at 7 PM tonight? Let me know if you are free.

type:ham
v2=URGENT SECURITY NOTICE: Your banking application has been temporarily locked due to suspicious activity. Please login here to verify your identity.

type:spam
